In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
    "passenger_id",
    "passenger_name",
    "city",
    "travel_class",
    "country"
]
df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)
display(df_day1)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]
df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)
display(df_day2)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
df_day1.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("passengers_delta")

In [0]:
display(spark.table("passengers_delta"))

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
spark.table("passengers_delta").count()

5

In [0]:
df = spark.read.format("delta").table("passengers_delta")
display(df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
%sql DESCRIBE HISTORY passengers_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T11:22:22Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(114506522194173),0617-091944-l0zs91nf,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1999)",null,Databricks-Runtime/17.3.x-photon-scala2.13


In [0]:
df_day2.createOrReplaceTempView("df_day2")

spark.sql("""
MERGE INTO passengers_delta AS target
USING df_day2 AS source
ON target.passenger_id = source.passenger_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("passengers_delta")
    .filter("passenger_id IN (102,104)")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India


In [0]:
display(
    spark.table("passengers_delta")
    .filter("passenger_id IN (106,107)")
)

passenger_id,passenger_name,city,travel_class,country
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
display(
    spark.table("passengers_delta")
    .filter("passenger_id = 102")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India


In [0]:
display(
    spark.table("passengers_delta")
    .filter("passenger_id = 106")
)

passenger_id,passenger_name,city,travel_class,country
106,Neha Singh,Pune,Economy,India


In [0]:
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("passengers_delta")

DataFrame[passenger_id: bigint, passenger_name: string, city: string, travel_class: string, country: string]

In [0]:
display(spark.table("passengers_delta"))

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
version0 = spark.read.format("delta") \
.option("versionAsOf",0) \
.table("passengers_delta")

latest = spark.table("passengers_delta")

display(version0)
display(latest)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
display(
version0.filter("passenger_id = 102")
)

display(
latest.filter("passenger_id = 102")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,Business,India


passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India


In [0]:
display(
version0.filter("passenger_id = 104")
)

display(
latest.filter("passenger_id = 104")
)

passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Delhi,Premium Economy,India


passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Hyderabad,Premium Economy,India


In [0]:
%sql
OPTIMIZE passengers_delta;

path,metrics
abfss://unity-catalog-storage@dbstorageimuuwwlwde2xc.dfs.core.windows.net/7405609256999242/__unitystorage/catalogs/397e9ec3-bb37-4774-aa90-697bbb227f1a/tables/96e27880-5382-4ad8-b2ab-c3852cfbf97b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781695673144, 1781695673740, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null)"


In [0]:
%sql
OPTIMIZE passengers_delta
ZORDER BY (city);

path,metrics
abfss://unity-catalog-storage@dbstorageimuuwwlwde2xc.dfs.core.windows.net/7405609256999242/__unitystorage/catalogs/397e9ec3-bb37-4774-aa90-697bbb227f1a/tables/96e27880-5382-4ad8-b2ab-c3852cfbf97b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2083), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781695684192, 1781695684534, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null)"


In [0]:
%sql
DELETE FROM passengers_delta
WHERE passenger_id = 105;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-17T11:28:19Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(114506522194173),0617-091944-l0zs91nf,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2083, p25FileSize -> 2059, numDeletionVectorsRemoved -> 1, minFileSize -> 2059, numAddedFiles -> 1, maxFileSize -> 2059, p75FileSize -> 2059, p50FileSize -> 2059, numAddedBytes -> 2059)",null,Databricks-Runtime/17.3.x-photon-scala2.13
3,2026-06-17T11:28:17Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#5942L = 105)""])",null,List(114506522194173),0617-091944-l0zs91nf,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1388, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1056, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 325)",null,Databricks-Runtime/17.3.x-photon-scala2.13
2,2026-06-17T11:25:47Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(114506522194173),0617-091944-l0zs91nf,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9098, p25FileSize -> 2083, numDeletionVectorsRemoved -> 1, minFileSize -> 2083, numAddedFiles -> 1, maxFileSize -> 2083, p75FileSize -> 2083, p50FileSize -> 2083, numAddedBytes -> 2083)",null,Databricks-Runtime/17.3.x-photon-scala2.13
1,2026-06-17T11:25:45Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#4613L = passenger_id#4602L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(114506522194173),0617-091944-l0zs91nf,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7099, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4752, materializeSourceTimeMs -> 139, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2635, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1909)",null,Databricks-Runtime/17.3.x-photon-scala2.13
0,2026-06-17T11:22:22Z,144642040528452,azuser7226_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(114506522194173),0617-091944-l0zs91nf,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1999)",null,Databricks-Runtime/17.3.x-photon-scala2.13


In [0]:
%sql
VACUUM passengers_delta;

path
abfss://unity-catalog-storage@dbstorageimuuwwlwde2xc.dfs.core.windows.net/7405609256999242/__unitystorage/catalogs/397e9ec3-bb37-4774-aa90-697bbb227f1a/tables/96e27880-5382-4ad8-b2ab-c3852cfbf97b
